# Logistic Regression — Simple Telco Churn Demo

## A beginner-friendly classroom walkthrough

### What are we trying to predict?

**Churn** means that a customer leaves or cancels the company's service.

Our goal is to use information about a customer to predict:

- `0` → No Churn
- `1` → Churn

For this first demo, we will deliberately keep the model **simple**.

We will use only **two features**:

1. `tenure` — how many months the customer has been with the company
2. `MonthlyCharges` — how much the customer is charged each month

Then we will use **Logistic Regression** to predict whether the customer churns.

> **Teaching idea:** Do not worry about building the "best" model yet. The purpose of this notebook is to understand the basic flow of classification.


# 1. The Big Picture

Before writing code, remember this simple machine learning workflow:

**DATA → FEATURES + TARGET → TRAIN/TEST → MODEL → PREDICTION → EVALUATION**

In this example:

**Features (X):**
- tenure
- MonthlyCharges

**Target (y):**
- Churn

The model learns patterns from customers whose outcomes we already know.


# 2. Upload and Load the Dataset

If you are running this notebook in Google Colab, run the cell below and upload your `Churn.csv` file.

If your file has a different name, the code will automatically use the first CSV file you upload.


In [1]:
from google.colab import files
import io
import pandas as pd

uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("File loaded:", filename)
print("Dataset shape:", df.shape)

df.head()


ModuleNotFoundError: No module named 'google'

### What should we look at first?

When we receive a dataset, we should not immediately build a model.

First ask:

- What does each row represent?
- What columns do we have?
- Which column is our target?
- Are there missing values?
- What kind of data do our features contain?

For this demo, our main question is:

> **Can tenure and monthly charges help us predict customer churn?**


In [ ]:
df[["tenure", "MonthlyCharges", "Churn"]].head(10)


# 3. Understand the Target

Let's look at how many customers churned and how many stayed.


In [ ]:
print(df["Churn"].value_counts())
print()
print(df["Churn"].value_counts(normalize=True) * 100)


### Reading the result

The `Churn` column contains:

- `Yes` → the customer left
- `No` → the customer stayed

This is a **binary classification problem** because there are only two possible outcomes.

Our model will eventually turn these into:

- `0` → No Churn
- `1` → Churn


# 4. Select Only Two Features

To keep our first demonstration easy to understand, we will use only:

- `tenure`
- `MonthlyCharges`

We will not use all the columns yet.

This lets us focus on **how Logistic Regression works** instead of getting distracted by preprocessing many categorical variables.


In [ ]:
X = df[["tenure", "MonthlyCharges"]].copy()

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())


# 5. Why Do We Convert Churn to 0 and 1?

Machine learning algorithms work with numerical values.

So we convert:

**No → 0**

**Yes → 1**

This gives the model a numerical target that represents the two classes.

> **Remember:** `0` and `1` are labels. They do not mean that "No" is mathematically half of "Yes." They simply represent the two classes.


# 6. Check for Missing Values

Before training, make sure the two features we selected do not contain missing values.


In [ ]:
print(X.isnull().sum())
print()
print(y.isnull().sum())


If missing values are present, they need to be handled before training.

For this simple demo, we will remove rows with missing values in the selected columns.


In [ ]:
data = pd.concat([X, y.rename("Churn")], axis=1).dropna()

X = data[["tenure", "MonthlyCharges"]]
y = data["Churn"]

print("Rows after cleaning:", len(data))


# 7. Split the Data

We do not want to train and test the model using exactly the same customers.

Instead:

**Training data → teaches the model**

**Testing data → checks whether the model can generalize to unseen customers**


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


### Why 80/20?

We are using:

- **80%** of the data for training
- **20%** for testing

The `random_state=42` makes our split reproducible.

The `stratify=y` argument helps keep the same proportion of churners and non-churners in both sets.


# 8. Train Logistic Regression

Now we finally build the model.

The model will learn the relationship between:

**tenure + MonthlyCharges → probability of churn**


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

print("Model trained successfully!")


# 9. What Did the Model Learn?

Logistic Regression learns a set of **coefficients (weights)**.

You can think of the model as asking:

> "How does each feature contribute to the likelihood of churn?"

Let's inspect the coefficients.


In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

coefficients


### How to interpret a coefficient

A **positive coefficient** means that, all else equal, increasing that feature pushes the model toward a higher churn probability.

A **negative coefficient** means that increasing that feature pushes the model toward a lower churn probability.

However, do not interpret the coefficient as proof that the feature *causes* churn.

The model is learning an association from the data.


# 10. Make Predictions

Now let's ask the model to predict the test customers.


In [ ]:
predictions = model.predict(X_test)

print(predictions[:20])


The output contains:

- `0` → predicted No Churn
- `1` → predicted Churn

But there is something more interesting than the final 0/1 prediction.

The model can also give us a **probability**.
predict() = “Just give me the final answer: 0 or 1.”




# 11. Get Churn Probabilities

Use `predict_proba()` to see the probability assigned to each class.
predict_proba() = “Tell me how confident/probable each class is.”

In [ ]:
probabilities = model.predict_proba(X_test)

print(probabilities[:10])


The two columns represent:

**Column 0 → Probability of No Churn**

**Column 1 → Probability of Churn**

So let's keep only the probability of Churn.


In [ ]:
churn_probability = model.predict_proba(X_test)[:, 1]

print(churn_probability[:10])


### Example interpretation

If the model gives:

`0.78`

we say:

> **The model estimates a 78% probability that this customer will churn.**

It does **not** mean the customer will definitely churn.

It is a probability estimate based on patterns learned from the training data.


# 12. Where Does the 0 or 1 Come From?

The model has probabilities, but classification needs a final class.

By default, we use a **0.50 threshold**.

**Probability ≥ 0.50 → Churn (1)**

**Probability < 0.50 → No Churn (0)**

This is how a probability becomes a classification.


In [ ]:
threshold = 0.50

predictions_from_probability = (
    churn_probability >= threshold
).astype(int)

print(predictions_from_probability[:20])


# 13. Let's Look at One Customer

Let's display the information for one test customer.

When I say X_test.iloc[0], I'm simply asking Python to show me the first customer in our test data. That particular customer happens to have 72 months of tenure and a monthly charge of 114.05.”


In [ ]:
example = X_test.iloc[0]

print("Customer information:")
print(example)

print("\nActual outcome:", y_test.iloc[0])

print(
    "\nPredicted churn probability:",
    round(churn_probability[0], 3)
)

print(
    "Predicted class:",
    predictions_from_probability[0]
)



> We gave the model this customer's tenure and monthly charges. The model uses what it learned from previous customers and produces a churn probability. We then compare that probability with our threshold to decide whether to classify the customer as a churner.

This is the basic idea of **classification with Logistic Regression**.


# 14. Evaluate the Model

A prediction is not enough. We need to ask:

> **How well did the model perform on customers it had never seen during training?**

We can start with accuracy.


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predictions_from_probability
)

print("Accuracy:", round(accuracy, 4))


### Accuracy

Accuracy answers:

> **"What percentage of all our predictions were correct?"**

For example, an accuracy of `0.80` means:

> The model correctly classified about **80% of the test customers**.

But remember:

**Accuracy is not the whole story**, especially when the classes are imbalanced.


# 15. Confusion Matrix

A confusion matrix helps us understand **what kinds of mistakes** the model made.


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    predictions_from_probability
)

print(cm)


The matrix follows this structure:

| | Predicted No Churn | Predicted Churn |
|---|---:|---:|
| **Actual No Churn** | TN | FP |
| **Actual Churn** | FN | TP |

### In simple words:

**TN — True Negative:** Customer stayed, and we correctly predicted they would stay.

**TP — True Positive:** Customer churned, and we correctly predicted they would churn.

**FP — False Positive:** We predicted churn, but the customer actually stayed.

**FN — False Negative:** We predicted the customer would stay, but they actually churned.

For a company trying to prevent churn, **False Negatives can be especially important** because those are customers we failed to identify before they left.


# 16. Classification Report

Instead of calculating precision, recall, and F1-score separately, we can generate a quick summary using `classification_report`.


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    predictions_from_probability,
    target_names=["No Churn", "Churn"],
    zero_division=0
))


### How to read the classification report

No Churn → the customer stays with the company.

Churn → the customer leaves/cancels the service.

**Precision**

> When the model predicts a class, how often is it correct?

**Recall**

> Of all the customers who actually belong to a class, how many did the model successfully find?

**F1-score**

> A balance between precision and recall.

**Support**

> How many actual examples of that class are in the test set?

The classification report is essentially a **report card for our classifier**.


# 17. A Quick Threshold Experiment

The 0.50 threshold is not a rule that can never be changed.

Let's see what happens if we lower it to **0.30**.

At 0.30, the model becomes more willing to call someone a potential churner.


In [ ]:
threshold = 0.30

predictions_30 = (
    churn_probability >= threshold
).astype(int)

print(classification_report(
    y_test,
    predictions_30,
    target_names=["No Churn", "Churn"],
    zero_division=0
))


### What should you notice?

When we lower the threshold:

- More customers are classified as potential churners.
- We usually catch more actual churners.
- **Recall tends to increase.**
- But we may also create more false positives.

This is a business decision.

If the company says:

> **"We really don't want to miss customers who are about to leave."**

then a lower threshold may be attractive.

If the company says:

> **"Retention offers are expensive, so only contact customers when we're quite confident."**

then a higher threshold may be more appropriate.


# 18. The Main Idea to Remember

Logistic Regression is a **classification algorithm**.

In this example:

**Input:**
- tenure
- MonthlyCharges

**Model:**
- Logistic Regression

**Output:**
- Probability of churn

**Decision:**
- Compare probability with a threshold

**Final class:**
- 0 → No Churn
- 1 → Churn

### The complete flow

**Customer Data → Logistic Regression → Churn Probability → Threshold → Prediction**
